# Recurrent Neural Networks and Transformers

This presentation covers the fundamentals of Recurrent Neural Networks (RNNs) and Transformer architectures.

<div class="slide-title">
    
# Recurrent Neural Networks and Transformers
    
</div>

## Overview

* Introduction: Sequential Data
* Recurrent Neural Networks (RNNs)
* LSTM and GRU
* Limitations of RNNs
* Transformer Architecture
* Attention Mechanism
* Comparison and Applications

## Introduction: Sequential Data

### What is Sequential Data?

<div class="group">
  <div class="text_70">

Sequential data is data where **order** matters:

* **Natural Language**: "The dog chases the cat" ≠ "The cat chases the dog"
* **Time Series**: Stock prices, Weather data, Sensor measurements
* **Audio and video**: Speech, Music, Motion sequences
* **DNA sequences**: Genetic information

The meaning depends on **context** and **temporal order**.

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/sequential_data.png">
  </div>
</div>

### Problem: Feed-Forward Networks

<div class="group">
  <div class="text_70">

**Why don't classical neural networks work for sequences?**

* Feed-Forward networks process each input **independently**
* **No memory** for previous inputs
* **Fixed input size** - but sequences have variable length
* **No temporal dependencies** between inputs

**Example**: Sentiment analysis of 'not good'
* Word by word: 'not' (negative), 'good' (positive)
* Together: 'not good' (negative!)

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/feedforward_limitation.png">
  </div>
</div>

## Recurrent Neural Networks (RNNs)

### The Basic Idea: Recurrence

<div class="group">
  <div class="text_70">

**RNNs introduce a 'memory':**

* The **hidden state** h_t stores information about previous inputs
* At each time step t:
  * Input: $x_t$ (current word/token)
  * Hidden state: $h_t = f(h_{t-1}, x_t)$
  * Output: $y_t = g(h_t)$

**Key Insight**: The same weights are shared across all time steps (Parameter Sharing)

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/rnn_basic.png">
  </div>
</div>

### RNN Architecture in Detail

**Mathematical Formulation:**

$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

$$y_t = W_{hy} h_t + b_y$$

**Where:**
* $W_{hh}$: weights for hidden-to-hidden
* $W_{xh}$: weights for input-to-hidden  
* $W_{hy}$: weights for hidden-to-output
* $b_h, b_y$: bias terms
* $\tanh$: activation function (alternatively: ReLU, sigmoid)

### RNN Unrolled Over Time

<div class="group">
  <div class="text_70">

**Unrolling**: The RNN can be represented as a chain

For a sequence 'I love machine learning':

* t=1: x₁='I' → h₁
* t=2: x₂='love' → h₂ (uses h₁)
* t=3: x₃='machine' → h₃ (uses h₂)
* t=4: x₄='learning' → h₄ (uses h₃)

Each hidden state 'remembers' all previous words.

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/rnn_unrolled.png">
  </div>
</div>

### Training: Backpropagation Through Time (BPTT)

<div class="group">
  <div class="text_70">

**How do you train an RNN?**

1. **Forward Pass**: Compute outputs for entire sequence
2. **Compute Loss**: Compare with labels
3. **Backward Pass**: Gradients flow **backwards through time**
4. **Update**: Adjust weights

**Problem**: For long sequences, gradients become very small or very large!

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/bptt.png">
  </div>
</div>

### The Vanishing Gradient Problem

<div class="group">
  <div class="text_70">

**Vanishing Gradients:**

* With long sequences (100+ steps), gradients become exponentially small
* Early time steps learn **poorly** or very slowly
* The network cannot learn **long-range dependencies**

**Exploding Gradients:**

* Gradients become exponentially large
* Training becomes unstable
* Solution: **Gradient Clipping**

  </div>
    </div>
</div>

## LSTM and GRU: Fortgeschrittene RNN-Architekturen

### LSTM: Long Short-Term Memory

<div class="group">
  <div class="text_70">

**The solution to vanishing gradients:**

LSTMs introduce a **cell state** $C_t$ - a separate 'data highway'

**Three gates control information flow:**

1. **Forget Gate** $(f_t)$: What should be forgotten?
2. **Input Gate** $(i_t)$: What new information is stored?
3. **Output Gate** $(o_t)$: What is output?

These gates enable selective remembering and forgetting.

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/lstm_cell.png">
  </div>
</div>

### LSTM Formulas

**Forget Gate:**
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**Input Gate:**
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$

**Cell State Update:**
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**Output Gate:**
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$

Where $\sigma$ = Sigmoid, $\odot$ = element-wise multiplication

### GRU: Gated Recurrent Unit

<div class="group">
  <div class="text_70">

**A simplified alternative to LSTM:**

* **Weniger Gates** (nur 2 statt 3)
* **No separate cell state**
* **Faster to train**
* Often **similar performance** to LSTM

**Two gates:**
1. **Update Gate** $(z_t)$: Wie viel vom alten State behalten?
2. **Reset Gate** $(r_t)$: Wie viel vom vorherigen State nutzen?

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/gru_cell.png">
  </div>
</div>

### Bidirectional RNNs

<div class="group">
  <div class="text_70">

**Use context from both directions:**

* **Forward RNN**: processes sequence from left to right
* **Backward RNN**: processes sequence from right to left
* **Combination**: $h_t = [h_t^forward; h_t^backward]$

**Example**: "The bank was full"
* Forward: 'The bank...' (Financial institution?)
* Backward: '...was full' (Park bench!)

**Disadvantage**: Requires the entire sequence (not suitable for real-time)

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/bidirectional_rnn.png">
  </div>
</div>

## Limitations of RNNs

### Why do we need something better?

<div class="group">
  <div class="text_70">

**Fundamental problems with RNNs/LSTMs:**

1. **Sequential processing**
   * Each time step depends on the previous one
   * **Keine Parallelisierung** möglich
   * Very slow with long sequences

2. **Information Bottleneck**
   * Entire sequence information in fixed hidden state
   * Even LSTM cannot perfectly model very long dependencies

3. **Computational cost**
   * O(n) sequential operations for sequence of length n

  </div>
  
</div>

### The central question

<div class="group">
  <div class="text">

## What if...

### ...we could consider all positions in the sequence **simultaneously**?

### ...each word could **directly** communicate with every other word?

### ...we could **decide ourselves** which parts of the sequence are important?

  </div>
</div>

<div style="text-align: center; font-size: 2em; margin-top: 50px;">

**→ Transformer!**

</div>

## Transformer Architecture

### 'Attention Is All You Need' (2017)

<div class="group">
  <div class="text_70">

**Revolutionary paper by Vaswani et al.:**

* **Kernidee**: Komplett ohne Rekurrenz!
* Instead: **Self-Attention mechanism**
* All positions are processed **in parallel**
* Direct connections between all token pairs

**Impact:**
* Basis for GPT, BERT, T5, ...
* Also in Computer Vision: Vision Transformer (ViT)
* State-of-the-art in almost all NLP tasks

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/attention_paper.png">
  </div>
</div>

## Self-Attention Mechanism

</div>
  <div class="images_30">
    <img src="../images/rnn_transformer/movie-scene.png">
  </div>
</div>

Image from "Eternal Sunshine of the Spotless Mind" - a movie about memory.
As humans, we are exceptionally good at detecting, which areas of the image are important and demand attention (e.g. the two people are more important than the left lower corner with only ice).
But a machine has to learn where to look.

### The Intuition Behind Attention

<div class="group">
  <div class="text_70">

**Example sentence**: "The robot didn't drop the cup because he was careful."

**Question**: What does "he" refer to?

* A human **pays attention to** relevant words: 'robot'
* Attention automatically learns **what to pay attention to**

**Self-Attention**: Each word can "communicate" with every other word

Attention weights show how strongly words are connected to each other.

</div>
  <div class="images_30">
    <img src="../images/rnn_transformer/attention_intuition.png">
  </div>
</div>

Showing the different attention paid to the other words in the sentence. Since cup is also an "it" it might receive a bit more attention as well, but in the end the most attention goes to "robot".

### Query, Key, Value - The Three Matrices

<div class="group">
  <div class="text_70">

**Analogy: Library**

* **Query (Q)**: "What am I searching for?" (Search query)
* **Key (K)**: "What do I offer?" (Book catalog)
* **Value (V)**: "What is the content?" (Book content)

**For each token x:**

* Q = x · W_Q (weight matrix for queries)
* K = x · W_K (weight matrix for keys)
* V = x · W_V (weight matrix for values)

These matrices are **learned** during training.

</div>
  <div class="images_30">
    <img src="../images/rnn_transformer/qkv.png">
  </div>
</div>

### Scaled Dot-Product Attention

**The Attention formula:**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**Step by step:**

1. **Compute scores**: $QK^T$ (How well does each query match each key?)
2. **Scale**: Division by $\sqrt{d_k}$ (Stability)
3. **Softmax**: Convert to probabilities
4. **Weighted sum**: Multiply with values

**Result**: For each token, a weighted combination of all other tokens!

### Attention

<div class="group">
  <div class="text_70">

**Example**: "The robot dropped the cup"

* 'dropped' pays strong attention to 'robot' (subject)
* 'dropped' pays attention to 'cup' (object)
* 'dropped' pays attention to 'the' (article)

Each word can simultaneously pay attention to **multiple** other words!
</div>

## Multi-Head Attention

### Why multiple attention heads?

<div class="group">
  <div class="text_70">

**Problem**: A single attention mechanism focuses on **one aspect**

**Solution**: Multiple parallel attention mechanisms ('heads')

**Different heads learn different relationships:**

* Head 1: Syntactic relationships (subject-verb)
* Head 2: Semantic relationships (pronoun reference)
* Head 3: Long-range dependencies
* Head 4: Local relationships (adjective-noun)

**Typical**: 8-16 heads in a transformer

      
  <div class="images_30">
    <img src="../images/rnn_transformer/movie-scene.png">
  </div>
</div>


Coming back to the image from earlier. It is not only the people in the image that are significant. Also the crack in the ice is a meaningful structure in the image (and a visual metaphor for the movie's message).
We thus can overlay multiple transformer heads to have it look at two distinct things in the image and thus capture more meaningful relations.

### Multi-Head Attention Formula

**For each head i:**

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Concatenation of all heads:**

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

**Where:**
* h = number of heads
* $W_i^Q, W_i^K, W_i^V$ = weight matrices for head i
* $W^O$ = output projection matrix

Each head has its **own** weight matrices!

## Full Transformer Architecture

### Positional Encoding

<div class="group">
  <div class="text_70">

**Problem**: Attention hat keine Information über **Reihenfolge**!

'I love cake' = 'Cake love I' (from attention's perspective)

**Lösung**: Positional Encoding

* Add **position information** to each token
* Sinusoidal functions of different frequencies

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$$

Each position receives a **unique** vector.

### Encoder Block

<div class="group">
  <div class="text_70">

**An encoder block consists of:**

1. **Multi-Head Self-Attention**
   * Each token pays attention to all others

2. **Add & Norm** (Residual Connection + Layer Normalization)
   * Stabilizes training of deep networks

3. **Feed-Forward Network**
   * Two linear layers with ReLU
   * Separate for each position

4. **Add & Norm** (once more)

**Typical**: 6-12 encoder blocks stacked

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/encoder_decoder.png">
  </div>
</div>

### Decoder Block

<div class="group">
  <div class="text_70">

**Decoder is similar to encoder, with additional components:**

1. **Masked Multi-Head Self-Attention**
   * Tokens can only pay attention to **previous** tokens
   * Prevents 'looking into the future'

2. **Add & Norm**

3. **Multi-Head Cross-Attention**
   * Decoder pays attention to **encoder output**
   * Q from decoder, K and V from encoder

4. **Add & Norm**

5. **Feed-Forward + Add & Norm**

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/encoder_decoder.png">
  </div>
</div>

### The Complete Transformer Architecture

<div class="group">
  <div class="text_70">

**Overall architecture:**

* **Input Embedding** + Positional Encoding
* **N Encoder Blocks** (typically 6)
* **N Decoder Blocks** (typically 6)
* **Linear Layer** + Softmax for Output

**Important properties:**

* Completely **parallelizable**
* Direct paths between all token pairs
* Scales well on GPUs/TPUs

  </div>
  <div class="images_30">
    <img src="../images/rnn_transformer/encoder_decoder.png">
  </div>
</div>

### Training and Efficiency

<div class="group">
  <div class="text_70">

**Advantages:**

* **Parallel Processing**: all tokens at once
* **Consistent Path Length**: O(1) instead of O(n) with RNNs
* **Better Gradients**: no vanishing gradients

**Challenges:**

* **Memory**: O(n²) for attention-matrix
* **Computational cost**: O(n²·d) for self-attention
* With long sequences (>1000 tokens) problematic

**Solution**: Efficient Transformers (Linformer, Performer, Longformer)


## Comparison: RNN vs. Transformer

### Direct Comparison

| **Aspekt** | **RNN/LSTM** | **Transformer** |
|------------|--------------|----------------|
| **Processing** | Sequential | Parallel |
| **Long-range dependencies** | Difficult (vanishing gradients) | Very good (direct connections) |
| **Training speed** | Slow (sequential) | Fast (parallelizable) |
| **Memory usage** | O(n) | O(n²) |
| **Computational cost** | O(n·d²) | O(n²·d) |
| **Short sequences** | Good | Overkill |
| **Long sequences** | Problematic | Very good (up to ~512 tokens) |
| **Interpretability** | Difficult | Good (Attention-Gewichte) |

### When to use what?

<div class="group">
  <div class="text_70">

**Use RNN/LSTM if:**

* Real-time applications (streaming data)
* Very long sequences (>10,000 time steps)
* Limited computational resources
* Simple sequential tasks
* Time Seriesvorhersage mit wenigen Features

**Use Transformer if:**

* Complex NLP tasks
* Sufficient computational resources available
* State-of-the-art performance desired
* Parallel training important
* Interpretability important (Attention-Visualization)


## Applications

### RNN/LSTM Applications

<div class="group">
  <div class="text_70">

**Classical applications:**

* **Speech recognition**: Speech-to-Text (in real-time)
* **Music Generation**: sequence of notes
* **Hand-writing recognition**: sequence of pen-strokes
* **Time Series Predictions**: 
  * Stock prices
  * Weather
  * Sensor data
* **Video-Analysis**: Frame-by-Frame Processing

RNNs are still relevant for **real-time streaming** and **resource-constrained** scenarios.


### Transformer Applications

<div class="group">
  <div class="text_70">

**Natural Language Processing:**

* **GPT (Generative)**: Text generation, ChatGPT
* **BERT (Bidirectional)**: Text classification, Q&A
* **T5 (Text-to-Text)**: Translation, Summarization

**Computer Vision:**

* **Vision Transformer (ViT)**: Image classification
* **DALL-E**: Text-to-image generation

**Other Domains:**

* **AlphaFold**: Protein structure prediction
* **Music Transformer**: Music composition
* **Code Generation**: GitHub Copilot
</div>

### Transformer Variants

**Encoder-only** (BERT-Style):
* Bidirectional context information
* Good für: Klassifikation, Named Entity Recognition, Q&A

**Decoder-only** (GPT-Style):
* Autoregressive text generation
* Good für: Text-Generierung, Dialog-Systeme

**Encoder-Decoder** (T5-Style):
* Complete sequence-to-sequence architecture
* Good für: Übersetzung, Summary, Text-zu-Text Tasks

## Current Developments and Outlook

If there is no time, the following part can be skipped.

### Efficient Transformers

<div class="group">
  <div class="text_70">

**Problem**: O(n²) memory and computational cost

**New approaches:**

* **Linformer**: Linear Attention with O(n) complexity
* **Performer**: Kernel-based approximation of attention
* **Longformer**: Sparse Attention for long documents
* **BigBird**: Combines local and global attention

**Goal**: Scaling to very long sequences (>10,000 tokens)


### Ethical Considerations and Bias

<div class="group">
  <div class="text_70">

**Challenges with large language models:**

* **Bias in training data**
  * Societal prejudices are reproduced
  * Gender bias, racist stereotypes

* **Lack of transparency**
  * 'Black box' despite attention visualization
  * Difficult to explain why certain decisions are made

* **Environmental impact**
  * Training large models (GPT-3) consumes enormous energy

**Important**: Critical reflection and responsible development!


### Research Questions

<div class="group">
  <div class="text_70">

**Open research questions:**

* Why do Transformers work **so well**?
* How can we better **interpret** attention mechanisms?
* Can Transformers learn **causal** relationships?
* How do **emergent abilities** form in large models?
* Are there more efficient alternatives to attention?

**Interdisciplinary perspective:**
* Connections to cognitive science
* Philosophical questions about 'understanding' and 'meaning'
* Social and ethical implications


## Summary

### Key Takeaways

**RNNs:**
* Sequential processing mit Hidden State as "memory"
* Vanishing Gradient Problem with long sequences
* LSTM/GRU partially solve this through gates
* Still relevant for real-time applications

**Transformer:**
* Parallelizable architecture based on self-attention
* Direct connections between all positions
* Multi-Head Attention for different relationship types
* Basis for state-of-the-art NLP models (GPT, BERT)

**Trade-off:** RNNs for efficiency, Transformers for performance

### Further Resources

**Papers:**
* Vaswani et al. (2017): "Attention Is All You Need"
* Hochreiter & Schmidhuber (1997): 'Long Short-Term Memory'
* Devlin et al. (2018): 'BERT: Pre-training of Deep Bidirectional Transformers'

**Online Resources:**
* The Illustrated Transformer (Jay Alammar)
* Stanford CS224N: Natural Language Processing with Deep Learning
* Hugging Face Transformers Documentation

**Hands-on:**
* Kaggle NLP Competitions
* Hugging Face Model Hub
* PyTorch/TensorFlow Tutorials